# Train toàn bộ thực nghiệm trên Kaggle (2 × T4)

5 mô hình × 4 mật độ × seed 42, 200 epoch, không dừng sớm. Hai GPU chạy song song:
GPU 0 = SPHR, LightGCN; GPU 1 = SGKR (CPU), XSimGCL, DirectAU.

**Cách chạy**
1. Settings: Accelerator **GPU T4 x2**, Internet **On**.
2. **Save Version → Save & Run All (Commit)**. Notebook chạy đến khi train xong (không tự giới hạn giờ) và lưu `results/`.
   Muốn tự dừng sớm thì đặt `TIME_BUDGET_H` (giờ) ở cell đầu; run đang dở vẫn tiếp tục được ở phiên sau.
3. Phiên sau: **Add Input → Your Work →** chọn output **mới nhất** của chính notebook này, rồi Save & Run All lại.
   Run đã xong được nạp lại và bỏ qua; run đang dở tiếp tục từ checkpoint của epoch gần nhất.
   Notebook ghim commit code của phiên đầu (`code_commit.txt`), nên push code mới không làm hỏng các run đang dở.
   Muốn chạy bằng code mới thì bắt đầu một notebook/output mới.
4. Cuối **mỗi** phiên, notebook vẽ hình và bảng từ các run đã xong vào `reports/`, hiện các hình chính ngay bên dưới
   và nén tất cả vào **`ket_qua.zip`** (metrics, history, summary, log, hình, bảng; không kèm checkpoint).
   Tải bằng link ở cell cuối hoặc ở tab Output. Khi đủ 20 run, đó là kết quả cuối cùng.


In [ ]:
import glob, os, shutil, signal, subprocess, sys, time
from pathlib import Path

REPO_URL = "https://github.com/tuyenda2011/advanced-machine.git"
CODE = Path("/tmp/advanced-machine")        # code + data (not kept in the output)
RESULTS = Path("/kaggle/working/results")   # kept as the notebook output
REPORTS = Path("/kaggle/working/reports")
TIME_BUDGET_H = None                        # no time limit; set hours (e.g. 11.3) to stop early and still build the report
EPOCHS = 200
WORKERS = {"0": ["sphr", "lightgcn"], "1": ["sgkr", "xsimgcl", "directau"]}
TOTAL_RUNS = 20
QUICK_CHECK = True                          # 1-epoch check of every command first (a few minutes)
start = time.time()

In [ ]:
previous_commit = sorted(glob.glob("/kaggle/input/**/code_commit.txt", recursive=True))
if not CODE.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(CODE)], check=True)
    if previous_commit:  # same code as the first session, or unfinished runs could not resume
        commit = Path(previous_commit[0]).read_text().strip()
        subprocess.run(["git", "fetch", "--depth", "1", "origin", commit], cwd=CODE, check=True)
        subprocess.run(["git", "checkout", "--detach", commit], cwd=CODE, check=True)
commit = subprocess.run(["git", "rev-parse", "HEAD"], cwd=CODE, capture_output=True, text=True, check=True).stdout.strip()
Path("/kaggle/working/code_commit.txt").write_text(commit + "\n")
print("code commit:", commit)
RESULTS.mkdir(parents=True, exist_ok=True)
# Runs of earlier sessions (this notebook's previous output added as an input): finished runs are
# skipped, unfinished runs continue from their per-epoch checkpoint (latest.pt / *_latest.pt).
for run in sorted(glob.glob("/kaggle/input/**/results/*/d*/seed*", recursive=True)):
    run = Path(run)                                  # .../results/<model>/d<pct>/seed<seed>
    target = RESULTS / run.relative_to(run.parents[2])
    if (target / "metrics.json").exists():
        continue
    if not target.exists() or (run / "metrics.json").exists():
        shutil.rmtree(target, ignore_errors=True)
        shutil.copytree(run, target)
done = sorted(str(p.parent.relative_to(RESULTS)) for p in RESULTS.glob("*/d*/seed*/metrics.json"))
unfinished = sorted(str(p.relative_to(RESULTS)) for p in RESULTS.glob("*/d*/seed*") if not (p / "metrics.json").exists())
print(f"{len(done)}/{TOTAL_RUNS} runs already done:", *done, sep="\n  ")
print(f"{len(unfinished)} unfinished run(s) will resume:", *unfinished, sep="\n  ")
subprocess.run(["nvidia-smi", "-L"])


In [ ]:
if QUICK_CHECK and len(done) < TOTAL_RUNS:
    subprocess.run([sys.executable, "scripts/smoke_test.py", "--density", "0.25", "--epochs", "1",
                    "--output", "/tmp/smoke"], cwd=CODE, check=True)

In [ ]:
deadline = start + TIME_BUDGET_H * 3600 if TIME_BUDGET_H else float("inf")
procs = {}
for gpu, models in WORKERS.items():
    log = open(f"/kaggle/working/worker_gpu{gpu}.log", "a")
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": gpu, "PYTHONUNBUFFERED": "1"}
    procs[gpu] = subprocess.Popen(
        [sys.executable, "scripts/run_experiments.py", "--models", *models, "--epochs", str(EPOCHS),
         "--device", "cuda", "--results", str(RESULTS), "--no-report"],
        cwd=CODE, env=env, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)

while any(p.poll() is None for p in procs.values()) and time.time() < deadline:
    time.sleep(300)
    finished = len(list(RESULTS.glob("*/d*/seed*/metrics.json")))
    print(f"{(time.time() - start) / 3600:4.1f} h: {finished}/{TOTAL_RUNS} runs done", flush=True)

for p in procs.values():                     # out of time: stop each worker and its training child
    if p.poll() is None:
        os.killpg(p.pid, signal.SIGTERM)
for p in procs.values():
    try:
        p.wait(120)
    except subprocess.TimeoutExpired:
        os.killpg(p.pid, signal.SIGKILL)

# Unfinished runs keep their checkpoints in the output; the next session continues them.
unfinished = sorted(str(p.relative_to(RESULTS)) for p in RESULTS.glob("*/d*/seed*") if not (p / "metrics.json").exists())
if unfinished:
    print("Unfinished, will resume next session:", *unfinished, sep="\n  ")

In [ ]:
import zipfile
from IPython.display import FileLink, Image, display

finished = sorted(str(p.parent.relative_to(RESULTS)) for p in RESULTS.glob("*/d*/seed*/metrics.json"))
print(f"{len(finished)}/{TOTAL_RUNS} runs done")
for gpu in WORKERS:
    print(f"--- worker_gpu{gpu}.log (last lines)")
    print("".join(open(f"/kaggle/working/worker_gpu{gpu}.log", encoding="utf-8", errors="replace").readlines()[-8:]))

if finished:
    # Figures and tables from whatever has finished (missing runs are marked "chưa chạy" on the plots).
    for script, extra in (("summarize_results.py", []), ("make_report.py", ["--output", str(REPORTS)])):
        if subprocess.run([sys.executable, f"scripts/{script}", "--results", str(RESULTS), *extra], cwd=CODE).returncode:
            print(f"{script} failed; the archive below still contains the raw results")
    # One small download: no checkpoints or prediction arrays (those stay in the Output tab).
    archive = Path("/kaggle/working/ket_qua.zip")
    with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zf:
        for item in [*RESULTS.glob("*/d*/seed*/metrics.json"), *RESULTS.glob("*/d*/seed*/history.csv"),
                     *RESULTS.glob("*.csv"), *RESULTS.glob("*.log")]:
            zf.write(item, Path("results") / item.relative_to(RESULTS))
        for item in REPORTS.rglob("*"):
            if item.is_file():
                zf.write(item, Path("reports") / item.relative_to(REPORTS))
        for item in Path("/kaggle/working").glob("worker_gpu*.log"):
            zf.write(item, Path("logs") / item.name)
    print(f"Archive: {archive} ({archive.stat().st_size / 2**20:.1f} MiB)")
    display(FileLink(archive.name))
    # Key figures inline: metric vs density, then NDCG@20 per density.
    figures = sorted((REPORTS / "figures").glob("H1_density_*.png")) + sorted((REPORTS / "figures").glob("H4_retention_*.png"))
    for figure in figures:
        print(figure.name)
        display(Image(filename=str(figure), width=720))

if len(finished) == TOTAL_RUNS:
    print("Xong: tải ket_qua.zip (hoặc results/ và reports/ ở tab Output).")
else:
    print("Chưa xong: Add Input -> output mới nhất của notebook này, rồi Save & Run All lại.")
